# Prefect — ETL de control contractual

Reinicie el kernel si se modificaron los módulos Python antes de ejecutar. Ejecute primero la celda que configura la ruta del proyecto.

La celda principal permite elegir una sola operación. Queda seleccionada **solo_postgres** para recuperar la carga fallida utilizando Silver y Gold existentes. Este modo no actualiza los datos desde la fuente ni recalcula Gold.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'secop_pipeline').exists():
    ROOT = ROOT.parent
if not (ROOT / 'secop_pipeline').exists():
    raise RuntimeError('Abra el notebook desde la carpeta del proyecto o notebooks')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
# ELIJA UN SOLO MODO antes de ejecutar esta celda.
# Reinicie el kernel si se actualizó load.py u otro módulo del proyecto.
# No lance otra ejecución mientras haya un ETL activo.

MODO ="reprocesar_local"  # Recuperación actual: Silver y Gold ya están generados.

# "solo_postgres": carga los archivos Silver y Gold existentes en PostgreSQL.
#   No descarga, no transforma y no recalcula indicadores.
#   Úselo cuando Gold terminó correctamente y falló únicamente la carga.
#   Si los archivos no cambian, volverá a cargar la misma información.
#   Es una carga directa: no crea un nuevo informe completo de Prefect
#   ni cambia el estado FALLIDA del informe de la ejecución anterior.
#
# "reprocesar_local": ejecuta el flujo con refresh=False.
#   Reutiliza Bronze, reutiliza Silver si su caché sigue vigente o lo transforma,
#   ejecuta calidad, RECONSTRUYE Gold y carga PostgreSQL.
#   Úselo para aplicar cambios de lógica a los datos ya descargados.
#   No incorpora datos nuevos de la API.
#
# "actualizar_todo": ejecuta el flujo con refresh=True.
#   Descarga nuevamente las cuatro bases; transforma o reutiliza Silver
#   según la caché, ejecuta calidad, reconstruye Gold y carga PostgreSQL.
#   Úselo para actualizar los datos desde la fuente. Es la opción más larga.
#   Revise las fechas configuradas y los parámetros de ircd_config.json.
#
# replace=True reemplaza las tablas al finalizar correctamente la carga.
# No agrega filas a las tablas anteriores; la carga es transaccional.

if MODO == "solo_postgres":
    from secop_pipeline.load import load_all_to_postgres
    resultado = load_all_to_postgres(replace=True)
elif MODO in ("reprocesar_local", "actualizar_todo"):
    from flows.secop_flow import secop_flow
    resultado = secop_flow(
        refresh=(MODO == "actualizar_todo"),
        load_postgres=True,
    )
else:
    raise ValueError(
        "MODO debe ser solo_postgres, reprocesar_local o actualizar_todo"
    )

resultado


## Cómo elegir la próxima ejecución

Cambie únicamente `MODO` en la celda anterior. Para recuperar la carga actual, mantenga `"solo_postgres"`. Para recalcular con datos locales, elija `"reprocesar_local"`. Para incorporar datos nuevos, elija `"actualizar_todo"`.

Los modos de flujo completo generan su informe de ejecución; la recuperación directa devuelve el resumen de carga en `resultado`. Las dos configuraciones Dask se ejecutan al construir Gold; no se ejecutan en `solo_postgres`.
